In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append("../src")
import config

files = sorted([f for f in os.listdir(config.RAW_DIR) if f.endswith(".csv")])
print(len(files), "files")

8 files


In [2]:
rows = []
for name in files:
    path = os.path.join(config.RAW_DIR, name)
    df = pd.read_csv(path, encoding="latin1", low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    empty_rows = int(df.isna().all(axis=1).sum())
    rows.append({
        "file": name,
        "rows": len(df),
        "empty_rows": empty_rows,
        "real_rows": len(df) - empty_rows,
        "columns": df.shape[1],
    })
summary = pd.DataFrame(rows)
summary

,file,rows,empty_rows,real_rows,columns
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,0,225745,85
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,0,286467,85
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,0,191033,85
3,Monday-WorkingHours.pcap_ISCX.csv,529918,0,529918,85
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,0,288602,85
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,458968,288602,170366,85
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,0,445909,85
7,Wednesday-workingHours.pcap_ISCX.csv,692703,0,692703,85


In [3]:
print("total rows:", summary["rows"].sum())
print("completely empty rows:", summary["empty_rows"].sum())
print("rows with data:", summary["real_rows"].sum())

total rows: 3119345
completely empty rows: 288602
rows with data: 2830743


In [4]:
path = os.path.join(config.RAW_DIR, files[0])
header = pd.read_csv(path, encoding="latin1", nrows=5)
raw_names = list(header.columns)
with_space = [c for c in raw_names if c != c.strip()]
print(len(raw_names), "columns,", len(with_space), "start or end with a space")
print("renamed duplicates:", [c for c in raw_names if c.strip().endswith(".1")])

85 columns, 70 start or end with a space
renamed duplicates: [' Fwd Header Length.1']


In [5]:
label_counts = {}
for name in files:
    path = os.path.join(config.RAW_DIR, name)
    labels = pd.read_csv(path, encoding="latin1", usecols=lambda c: c.strip() == "Label")
    labels.columns = ["Label"]
    counts = labels["Label"].value_counts()
    for value in counts.index:
        label_counts[value] = label_counts.get(value, 0) + int(counts[value])
for value in label_counts:
    print(repr(value), label_counts[value])
print("sum of label counts:", sum(label_counts.values()))

C:\Users\laksh\AppData\Local\Temp\ipykernel_101424\2019712703.py:4: DtypeWarning: Columns (84) have mixed types. Specify dtype option on import or set low_memory=False.
  labels = pd.read_csv(path, encoding="latin1", usecols=lambda c: c.strip() == "Label")


'DDoS' 128027
'BENIGN' 2273097
'PortScan' 158930
'Bot' 1966
'Infiltration' 36
'Web Attack \x96 Brute Force' 1507
'Web Attack \x96 XSS' 652
'Web Attack \x96 Sql Injection' 21
'FTP-Patator' 7938
'SSH-Patator' 5897
'DoS Hulk' 231073
'DoS GoldenEye' 10293
'DoS slowloris' 5796
'DoS Slowhttptest' 5499
'Heartbleed' 11
sum of label counts: 2830743


In [6]:
for name in files:
    path = os.path.join(config.RAW_DIR, name)
    stamps = pd.read_csv(path, encoding="latin1", usecols=lambda c: c.strip() == "Timestamp")
    stamps.columns = ["Timestamp"]
    values = stamps["Timestamp"].dropna()
    parsed = pd.to_datetime(values, format="%d/%m/%Y %H:%M", errors="coerce")
    print(name)
    print("   sample:", list(values.head(3)))
    print("   not matching day/month/year hour:minute:", int(parsed.isna().sum()))
    print("   days:", sorted(parsed.dt.day.dropna().unique().tolist()))
    print("   hours:", sorted(parsed.dt.hour.dropna().unique().tolist()))

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
   sample: ['7/7/2017 3:30', '7/7/2017 3:30', '7/7/2017 3:30']
   not matching day/month/year hour:minute: 0
   days: [7]
   hours: [3, 4, 5]
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
   sample: ['7/7/2017 1:00', '7/7/2017 1:00', '7/7/2017 1:00']
   not matching day/month/year hour:minute: 0
   days: [7]
   hours: [1, 2, 3]
Friday-WorkingHours-Morning.pcap_ISCX.csv
   sample: ['7/7/2017 8:59', '7/7/2017 8:59', '7/7/2017 9:00']
   not matching day/month/year hour:minute: 0
   days: [7]
   hours: [8, 9, 10, 11, 12]
Monday-WorkingHours.pcap_ISCX.csv
   sample: ['03/07/2017 08:55:58', '03/07/2017 08:55:58', '03/07/2017 08:55:58']
   not matching day/month/year hour:minute: 529918
   days: []
   hours: []
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
   sample: ['6/7/2017 1:00', '6/7/2017 1:00', '6/7/2017 1:00']
   not matching day/month/year hour:minute: 0
   days: [6]
   hours: [1, 2, 3, 4, 5]


C:\Users\laksh\AppData\Local\Temp\ipykernel_101424\3286455354.py:3: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  stamps = pd.read_csv(path, encoding="latin1", usecols=lambda c: c.strip() == "Timestamp")


Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
   sample: ['6/7/2017 8:59', '6/7/2017 8:59', '6/7/2017 8:59']
   not matching day/month/year hour:minute: 0
   days: [6]
   hours: [8, 9, 10, 11, 12]
Tuesday-WorkingHours.pcap_ISCX.csv
   sample: ['4/7/2017 8:54', '4/7/2017 8:54', '4/7/2017 8:54']
   not matching day/month/year hour:minute: 0
   days: [4]
   hours: [1, 2, 3, 4, 5, 8, 9, 10, 11, 12]
Wednesday-workingHours.pcap_ISCX.csv
   sample: ['5/7/2017 8:42', '5/7/2017 8:42', '5/7/2017 8:42']
   not matching day/month/year hour:minute: 0
   days: [5]
   hours: [1, 2, 3, 4, 5, 8, 9, 10, 11, 12]


In [7]:
nan_counts = {}
inf_counts = {}
negative_counts = {}
for name in files:
    path = os.path.join(config.RAW_DIR, name)
    df = pd.read_csv(path, encoding="latin1", low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna(how="all")
    for col in df.columns:
        if col in config.NON_FEATURE_COLUMNS:
            continue
        values = pd.to_numeric(df[col], errors="coerce")
        nan_counts[col] = nan_counts.get(col, 0) + int(values.isna().sum())
        inf_counts[col] = inf_counts.get(col, 0) + int(np.isinf(values).sum())
        negative_counts[col] = negative_counts.get(col, 0) + int((values < 0).sum())
audit = pd.DataFrame({"nan": nan_counts, "inf": inf_counts, "negative": negative_counts})
audit[(audit > 0).any(axis=1)].sort_values("nan", ascending=False)

,nan,inf,negative
Flow Bytes/s,1358,1509,85
Flow Duration,0,0,115
Flow Packets/s,0,2867,115
Flow IAT Mean,0,0,115
Flow IAT Max,0,0,115
Flow IAT Min,0,0,2891
Fwd IAT Min,0,0,17
Fwd Header Length,0,0,35
Bwd Header Length,0,0,22
Fwd Header Length.1,0,0,35


In [8]:
duplicate_total = 0
for name in files:
    path = os.path.join(config.RAW_DIR, name)
    df = pd.read_csv(path, encoding="latin1", low_memory=False)
    df = df.dropna(how="all")
    count = int(df.duplicated().sum())
    print(name, count)
    duplicate_total = duplicate_total + count
print("exact duplicate rows inside files:", duplicate_total)

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv 2
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv 1
Friday-WorkingHours-Morning.pcap_ISCX.csv 2
Monday-WorkingHours.pcap_ISCX.csv 34
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv 142
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv 1
Tuesday-WorkingHours.pcap_ISCX.csv 4
Wednesday-workingHours.pcap_ISCX.csv 17
exact duplicate rows inside files: 203
